# 🩺 HealSense - Google Colab Setup

This notebook sets up the HealSense project in Google Colab and downloads all necessary datasets.

**What this notebook does:**
- Clone the GitHub repository
- Install required packages
- Setup API credentials (Kaggle, OpenAI/Gemini)
- Download all datasets (UCI, Kaggle, Synthetic)
- Verify installation

---

## 🚀 Step 1: Clone Repository

In [ ]:
import os
from pathlib import Path

# Clone the repository
!git clone https://github.com/YOUR_USERNAME/FYP-Project.git
%cd FYP-Project/healsense

print("✅ Repository cloned successfully!")
print(f"📍 Current directory: {os.getcwd()}")

## 📦 Step 2: Install Dependencies

In [ ]:
# Install required packages
!pip install -q -r requirements.txt

# Additional packages for Colab
!pip install -q kaggle python-dotenv google-generativeai

print("✅ All dependencies installed!")

## 🔑 Step 3: Setup API Credentials

### 3.1 Kaggle API (Required for datasets)

**Get your Kaggle API key:**
1. Go to https://www.kaggle.com/account
2. Scroll to "API" section
3. Click "Create New API Token"
4. Download `kaggle.json`
5. Upload it in the cell below

In [ ]:
# Upload kaggle.json file
from google.colab import files
import json

print("📤 Please upload your kaggle.json file:")
uploaded = files.upload()

# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✅ Kaggle API configured!")

### 3.2 Google Gemini API (Optional - for AI features)

Get your Gemini API key from: https://makersuite.google.com/app/apikey

In [ ]:
import os
from getpass import getpass

# Enter your Gemini API key (optional)
gemini_api_key = getpass("Enter your Google Gemini API key (or press Enter to skip): ")

if gemini_api_key:
    os.environ['GEMINI_API_KEY'] = gemini_api_key
    print("✅ Gemini API key configured!")
else:
    print("⏭️  Skipped Gemini API setup")

### 3.3 Create .env file

In [ ]:
# Create .env file with credentials
env_content = f"""
# Kaggle API (configured)
KAGGLE_CONFIGURED=true

# Google Gemini API
GEMINI_API_KEY={os.environ.get('GEMINI_API_KEY', 'your_gemini_api_key_here')}

# Database (SQLite for Colab)
DATABASE_URL=sqlite:///healsense.db

# Security
SECRET_KEY=colab_development_key_not_for_production

# Flask
FLASK_ENV=development
DEBUG=True
"""

with open('.env', 'w') as f:
    f.write(env_content)

print("✅ .env file created!")

## 📊 Step 4: Download Datasets

This will download:
- UCI Heart Disease Dataset (~920 records)
- Kaggle Health Datasets (200K+ records)
- Generate Synthetic Data (10K records)

In [ ]:
# Run the master download script
!python scripts/download_all_datasets.py --skip-physionet

print("\n✅ Dataset download complete!")

## 🔍 Step 5: Verify Installation

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path('data/raw')

print("📂 Checking downloaded datasets:\n")

# Check UCI Heart Disease
uci_dir = data_dir / 'uci_heart_disease'
if uci_dir.exists():
    files = list(uci_dir.glob('*.data'))
    print(f"✅ UCI Heart Disease: {len(files)} files")
else:
    print("❌ UCI Heart Disease: Not found")

# Check Kaggle datasets
kaggle_dir = data_dir / 'kaggle_health_data'
if kaggle_dir.exists():
    csv_files = list(kaggle_dir.rglob('*.csv'))
    print(f"✅ Kaggle Datasets: {len(csv_files)} CSV files")
    
    # Try to load one dataset
    if csv_files:
        sample_file = csv_files[0]
        df = pd.read_csv(sample_file, nrows=5)
        print(f"   Sample file: {sample_file.name}")
        print(f"   Columns: {list(df.columns)[:5]}...")
else:
    print("⚠️  Kaggle Datasets: Not found (check Kaggle API setup)")

# Check Synthetic data
synthetic_file = data_dir / 'synthetic' / 'synthetic_vital_signs.csv'
if synthetic_file.exists():
    df_synthetic = pd.read_csv(synthetic_file)
    print(f"✅ Synthetic Data: {len(df_synthetic)} records")
    print(f"   Columns: {list(df_synthetic.columns)}")
else:
    print("❌ Synthetic Data: Not found")

print("\n" + "="*60)
print("Setup verification complete!")
print("="*60)

## 📈 Step 6: Quick Data Exploration

Let's explore the synthetic vital signs data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Load synthetic data
df = pd.read_csv('data/raw/synthetic/synthetic_vital_signs.csv')

print("📊 Synthetic Vital Signs Dataset")
print(f"\nShape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
print(df.head())

# Distribution of alert types
print(f"\n🚨 Alert Distribution:")
print(df['alert_type'].value_counts())

# Plot vital signs distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Vital Signs Distributions', fontsize=16)

vital_signs = ['heart_rate', 'spo2', 'temperature', 'systolic_bp', 'diastolic_bp']
for idx, vital in enumerate(vital_signs):
    ax = axes[idx // 3, idx % 3]
    df[vital].hist(bins=50, ax=ax, color='skyblue', edgecolor='black')
    ax.set_title(vital.replace('_', ' ').title())
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')

# Alert distribution pie chart
ax = axes[1, 2]
df['alert_type'].value_counts().plot(kind='pie', ax=ax, autopct='%1.1f%%')
ax.set_title('Alert Type Distribution')
ax.set_ylabel('')

plt.tight_layout()
plt.show()

print("\n✅ Exploration complete!")

## 🎯 Step 7: Mount Google Drive (Optional)

Mount Google Drive to save your work permanently

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create a directory for HealSense in Drive
drive_dir = Path('/content/drive/MyDrive/HealSense')
drive_dir.mkdir(exist_ok=True)

print(f"✅ Google Drive mounted at: {drive_dir}")
print("\nYou can now copy important files to Drive for permanent storage")

## 📝 Next Steps

Now you can:

1. **Explore the data** - Open `notebooks/01_data_exploration.ipynb`
2. **Train models** - Run ML experiments
3. **Test APIs** - Explore backend functionality
4. **Develop features** - Work on new functionality

### Quick Commands

```bash
# List all datasets
!ls -lh data/raw/*/

# Check total data size
!du -sh data/

# View download report
!cat DOWNLOAD_REPORT.md
```

---

### 🆘 Troubleshooting

**Kaggle API not working?**
- Make sure you uploaded `kaggle.json`
- Check permissions: `!ls -la ~/.kaggle/`

**Datasets not downloading?**
- Check internet connection
- Verify Kaggle API token is valid
- Try running download script again

**Need help?**
- Check the README.md
- Review documentation in docs/
- Open an issue on GitHub

---

**Happy Coding! 🚀**